In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
BASE_PATH = "/Volumes/main/lakehouse_marketing"
RAW_PATH = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"

%md
* Definindo Schema e lendo o csv da Raw

In [0]:
events_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("campaign_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", StringType(), True)
])


df_events_raw = spark.read\
                    .schema(events_schema)\
                    .option("header", "true")\
                    .csv(f"{RAW_PATH}/events")

In [0]:
df_events_bronze = df_events_raw\
                        .withColumn("ingestion_timestamp", F.current_timestamp())\
                        .withColumn("source_file", F.col("_metadata.file_path"))


In [0]:
display(df_events_bronze.limit(5))

In [0]:
BRONZE_EVENTS_PATH = f"{BRONZE_PATH}/events"

df_events_bronze.write\
    .format('delta')\
    .mode('overwrite')\
    .save(BRONZE_EVENTS_PATH)

In [0]:
dbutils.fs.ls(BRONZE_EVENTS_PATH)

In [0]:
display(spark.read\
    .format("delta")\
    .load(f"{BRONZE_EVENTS_PATH}"))

In [0]:
spark.read\
    .format("delta")\
    .load(f"{BRONZE_EVENTS_PATH}")\
    .count()